In [ ]:
#KAGGLE CLIENT
import os
import pandas as pd
from pathlib import Path

class KaggleDataFetcher:
    def __init__(self):
        self.dataset_name = "bittlingmayer/amazonreviews"
        self.download_path = Path("./data")
        self.download_path.mkdir(exist_ok=True)
        self.authenticated = False
        # Look for kaggle.json in standard location
        self.kaggle_config_dir = Path.home() / '.kaggle'
        self.kaggle_config_file = self.kaggle_config_dir / 'kaggle.json'

    def authenticate(self):
        """
        Configure Kaggle credentials from kaggle.json file in user's home directory.
        Will NOT raise if credentials are missing — falls back to offline/sample mode.
        """
        if not self.kaggle_config_file.exists():
            print(f"Warning: Kaggle credentials not found at {self.kaggle_config_file}")
            print("To add credentials:")
            print("1. Go to https://www.kaggle.com/settings/account")
            print("2. Click 'Create New API Token' to download kaggle.json")
            print("3. Create directory: %USERPROFILE%\\.kaggle")
            print("4. Move kaggle.json to %USERPROFILE%\\.kaggle\\kaggle.json")
            self.authenticated = False
            return

        try:
            import json
            with open(self.kaggle_config_file) as f:
                credentials = json.load(f)
                
            os.environ["KAGGLE_USERNAME"] = credentials.get("username")
            os.environ["KAGGLE_KEY"] = credentials.get("key")
            
            # Verify credentials format
            if not credentials.get("username") or not credentials.get("key"):
                raise ValueError("Invalid credentials format in kaggle.json")
                
            self.authenticated = True
            print("Kaggle authentication configured successfully.")
            
        except Exception as e:
            print(f"Warning: Failed to load Kaggle credentials: {e}")
            self.authenticated = False
            
    def download_bittlingmayer/amazonreviews_dataset():
        """Download bittlingmayer/amazonreviews dataset from Kaggle using API"""
        print("Downloading bittlingmayer/amazonreviews dataset from Kaggle...")

        kaggle.api.dataset_download_files(
            'uciml/bittlingmayer/amazonreviews',
            path='./data',
            unzip=True
        )

    print("Dataset downloaded successfully!")
    return './data/bittlingmayer/amazonreviews.csv'

    def load_reviews(self, file_pattern="*.csv", limit=None):
        import kagglehub
        """
        Load reviews from downloaded CSV files.
        Returns a pandas DataFrame.
        """
        csv_files = list(self.download_path.glob(file_pattern))

        if not csv_files:
            raise FileNotFoundError(
                f"No CSV files found in {self.download_path}. Please download the dataset first."
            )

        print(f"Found {len(csv_files)} CSV file(s).")

        dataframes = []
        for csv_file in csv_files:
            print(f"Loading {csv_file.name}...")
            try:
                df = pd.read_csv(csv_file, nrows=limit)
                dataframes.append(df)
            except Exception as e:
                print(f"Error loading {csv_file.name}: {str(e)}")

        if not dataframes:
            raise Exception("Failed to load any data from CSV files.")

        combined_df = pd.concat(dataframes, ignore_index=True)
        print(f"Loaded {len(combined_df)} reviews total.")

        return combined_df

    def fetch_and_load(self, limit=None, skip_download=False):
        """
        Convenience method to authenticate, download (if possible), and load reviews.
        If no Kaggle credentials / package / CSVs are present, returns a small sample DataFrame.
        """
        self.authenticate()

        # Attempt to download only if authenticated
        if self.authenticated:
            if not skip_download or not any(self.download_path.glob("*.csv")):
                self.download_dataset()
            else:
                print("Skipping download, using existing data.")
        else:
            print("Running without Kaggle download (offline/sample mode).")

        # Try to load real data; if not available, return sample DataFrame
        try:
            df = self.load_reviews(limit=limit)
            # normalize common column names to expected ones used by process_reviews
            # prefer 'reviewText' or fallback to 'review' or 'text'
            if 'reviewText' not in df.columns and 'review' in df.columns:
                df = df.rename(columns={'review': 'reviewText'})
            if 'reviewText' not in df.columns and 'text' in df.columns:
                df = df.rename(columns={'text': 'reviewText'})
            return df
        except FileNotFoundError:
            print("No dataset found locally. Returning sample reviews.")
        except Exception as e:
            print(f"Error loading dataset: {e}. Returning sample reviews.")

        # Fallback sample DataFrame
        sample_reviews = [
            "The Apple iPhone 13 is amazing! Great battery life and sleek design. Highly recommend!",
            "Terrible experience with Samsung Galaxy S21. Poor customer service from Samsung. Would not buy again.",
            "Sony WH-1000XM4 headphones are excellent. Best headphones for music lovers. Worth the price!",
            "Disappointed with Dell laptop. Keyboard stopped working after 2 months. Avoid Dell products.",
            "Google Pixel 6 Pro camera is outstanding. Perfect phone for photography enthusiasts.",
            "Worst product ever from Amazon Basics. Broke within a week. Waste of money."
        ]
        df_sample = pd.DataFrame({
            'review_id': range(1, len(sample_reviews) + 1),
            'reviewText': sample_reviews,
            'overall': [5, 1, 5, 1, 5, 1]
        })
        return df_sample
    


# MAIN APP
#!/usr/bin/env python3
"""
Amazon Reviews Sentiment Analysis Application

This application:
1. Fetches Amazon reviews from Kaggle dataset
2. Performs Named Entity Recognition (NER) to extract product names and brands
3. Analyzes sentiment using a rule-based approach
4. Stores results in Supabase database
"""

import os
import sys
from dotenv import load_dotenv
# Using classes defined in this file directly instead of importing separate modules.
# The following classes are defined above in this same file:
#   KaggleDataFetcher, NERExtractor, RuleBasedSentimentAnalyzer, DatabaseHandler


def print_separator():
    """Print a visual separator."""
    print("\n" + "=" * 80 + "\n")


def process_reviews(df, ner_extractor, sentiment_analyzer, db_handler, limit=None):
    """
    Process reviews: extract entities, analyze sentiment, and store in database.

    Args:
        df: DataFrame containing reviews
        ner_extractor: NERExtractor instance
        sentiment_analyzer: RuleBasedSentimentAnalyzer instance
        db_handler: DatabaseHandler instance
        limit: Maximum number of reviews to process
    """
    if limit:
        df = df.head(limit)

    print(f"Processing {len(df)} reviews...")
    print_separator()

    processed_count = 0
    skipped_count = 0

    for idx, row in df.iterrows():
        try:
            review_text = str(row.get('reviewText', row.get('review', '')))

            if not review_text or len(review_text.strip()) == 0:
                continue

            review_id = f"kaggle_{idx}"

            if db_handler.review_exists(review_id):
                skipped_count += 1
                continue

            rating = row.get('overall', row.get('rating', None))
            product_name = str(row.get('asin', ''))

            review_data = {
                'review_id': review_id,
                'product_name': product_name,
                'review_text': review_text,
                'rating': int(rating) if rating else None
            }

            review_uuid = db_handler.insert_review(review_data)

            entities_result = ner_extractor.extract_products_and_brands(review_text)
            entities = []
            for product in entities_result['products']:
                entities.append({'entity_type': 'PRODUCT', 'entity_text': product})
            for brand in entities_result['brands']:
                entities.append({'entity_type': 'BRAND', 'entity_text': brand})

            if entities:
                db_handler.insert_entities(review_uuid, entities)

            sentiment_result = sentiment_analyzer.analyze(review_text)
            db_handler.insert_sentiment(review_uuid, sentiment_result)

            processed_count += 1

            if processed_count <= 5:
                print(f"\n[Review #{processed_count}]")
                print(f"Text: {review_text[:100]}...")
                print(f"Extracted Entities:")
                if entities_result['products']:
                    print(f"  - Products: {', '.join(entities_result['products'][:3])}")
                if entities_result['brands']:
                    print(f"  - Brands: {', '.join(entities_result['brands'][:3])}")
                if not entities:
                    print(f"  - No entities found")
                print(f"Sentiment Analysis:")
                print(f"  - Sentiment: {sentiment_result['sentiment'].upper()}")
                print(f"  - Confidence: {sentiment_result['confidence_score']}")
                if sentiment_result['positive_words']:
                    print(f"  - Positive words: {', '.join(sentiment_result['positive_words'][:5])}")
                if sentiment_result['negative_words']:
                    print(f"  - Negative words: {', '.join(sentiment_result['negative_words'][:5])}")
                print("-" * 80)
            elif processed_count % 10 == 0:
                print(f"Processed {processed_count} reviews...")

        except Exception as e:
            print(f"Error processing review {idx}: {str(e)}")
            continue

    print_separator()
    print(f"Processing complete!")
    print(f"  - Processed: {processed_count} reviews")
    print(f"  - Skipped (already exist): {skipped_count} reviews")


def display_statistics(db_handler):
    """
    Display sentiment statistics from the database.

    Args:
        db_handler: DatabaseHandler instance
    """
    print_separator()
    print("SENTIMENT STATISTICS")
    print_separator()

    stats = db_handler.get_sentiment_statistics()
    total = sum(stats.values())

    if total == 0:
        print("No sentiment data available yet.")
        return

    print(f"Total reviews analyzed: {total}")
    print(f"\nSentiment breakdown:")
    print(f"  Positive: {stats['positive']} ({stats['positive']/total*100:.1f}%)")
    print(f"  Negative: {stats['negative']} ({stats['negative']/total*100:.1f}%)")
    print(f"  Neutral:  {stats['neutral']} ({stats['neutral']/total*100:.1f}%)")


def display_sample_results(db_handler, num_samples=5):
    """
    Display sample reviews with their analysis.

    Args:
        db_handler: DatabaseHandler instance
        num_samples: Number of sample reviews to display
    """
    print_separator()
    print(f"SAMPLE RESULTS FROM DATABASE (showing {num_samples} reviews)")
    print_separator()

    reviews = db_handler.get_reviews_with_analysis(limit=num_samples)

    if not reviews:
        print("No reviews found in database.")
        return

    for i, review in enumerate(reviews, 1):
        print(f"\n[Review #{i}]")
        print(f"Rating: {review.get('rating', 'N/A')} stars")
        print(f"Review Text: {review['review_text'][:200]}...")
        print()

        entities = db_handler.get_entities_by_review(review['id'])
        print("EXTRACTED ENTITIES:")
        if entities:
            products = [e['entity_text'] for e in entities if e['entity_type'] == 'PRODUCT']
            brands = [e['entity_text'] for e in entities if e['entity_type'] == 'BRAND']
            if products:
                print(f"  Products: {', '.join(products)}")
            if brands:
                print(f"  Brands: {', '.join(brands)}")
            if not products and not brands:
                print(f"  No entities extracted")
        else:
            print("  No entities extracted")
        print()

        if review.get('sentiment_analysis') and len(review['sentiment_analysis']) > 0:
            sentiment = review['sentiment_analysis'][0]
            print("SENTIMENT ANALYSIS:")
            print(f"  Sentiment: {sentiment['sentiment'].upper()}")
            print(f"  Confidence Score: {sentiment['confidence_score']}")
            if sentiment.get('positive_words') and len(sentiment['positive_words']) > 0:
                print(f"  Positive Words Found: {', '.join(sentiment['positive_words'])}")
            if sentiment.get('negative_words') and len(sentiment['negative_words']) > 0:
                print(f"  Negative Words Found: {', '.join(sentiment['negative_words'])}")
        else:
            print("SENTIMENT ANALYSIS:")
            print("  No sentiment data available")

        print("-" * 80)


def main():
    """Main application entry point."""
    load_dotenv()

    print_separator()
    print("AMAZON REVIEWS SENTIMENT ANALYSIS")
    print_separator()

    print("Initializing components...")

    try:
        kaggle_fetcher = KaggleDataFetcher()
        ner_extractor = NERExtractor()
        sentiment_analyzer = RuleBasedSentimentAnalyzer()
        db_handler = DatabaseHandler()
    except Exception as e:
        print(f"\nError during initialization: {str(e)}")
        print("\nPlease ensure:")
        print("1. Required Python packages are installed: pip install -r requirements.txt")
        print("2. spaCy English model is installed: python -m spacy download en_core_web_sm")
        sys.exit(1)

    print("Components initialized successfully!")
    print_separator()

    # Automatically set to process 50 reviews
    num_reviews = 50
    print(f"\nAutomatically processing {num_reviews} reviews...")

    try:
        df = kaggle_fetcher.fetch_and_load(limit=num_reviews, skip_download=False)
    except Exception as e:
        print(f"\nError fetching data: {str(e)}")
        print("\nFalling back to sample dataset.")
        # Create sample DataFrame if fetch fails
        df = pd.DataFrame({
            'reviewText': sample_reviews,
            'overall': [5, 1, 5, 1, 5, 1]
        })

    process_reviews(df, ner_extractor, sentiment_analyzer, db_handler, limit=num_reviews)

    display_statistics(db_handler)
    display_sample_results(db_handler, num_samples=3)

    print_separator()
    print("Application completed successfully!")
    print("All data has been saved to memory/database.")
    print_separator()




#NER EXTRACTOR
import spacy
from typing import List, Dict, Set


class NERExtractor:
    """
    Extracts named entities (product names, brands) from review text using spaCy.
    """

    def __init__(self, model_name="en_core_web_sm"):
        """
        Initialize the NER extractor with a spaCy model.

        Args:
            model_name: Name of the spaCy model to use
        """
        try:
            self.nlp = spacy.load(model_name)
            print(f"Loaded spaCy model: {model_name}")
        except OSError:
            print(f"Model {model_name} not found. Downloading...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", model_name])
            self.nlp = spacy.load(model_name)
            print(f"Downloaded and loaded spaCy model: {model_name}")

        self.product_keywords = {
            'phone', 'laptop', 'tablet', 'computer', 'camera', 'headphones',
            'speaker', 'watch', 'tv', 'monitor', 'keyboard', 'mouse',
            'charger', 'case', 'cover', 'cable', 'adapter', 'battery',
            'book', 'kindle', 'echo', 'alexa', 'fire', 'prime'
        }

    def extract_entities(self, text: str) -> List[Dict[str, str]]:
        """
        Extract named entities from text.

        Args:
            text: The review text to analyze

        Returns:
            List of dictionaries containing entity_type and entity_text
        """
        if not text or not isinstance(text, str):
            return []

        doc = self.nlp(text)
        entities = []

        for ent in doc.ents:
            if ent.label_ in ['PRODUCT', 'ORG', 'PERSON', 'GPE']:
                entities.append({
                    'entity_type': ent.label_,
                    'entity_text': ent.text.strip()
                })

        return entities

    def extract_products_and_brands(self, text: str) -> Dict[str, List[str]]:
        """
        Extract product names and brands from review text using NER and rule-based methods.

        Args:
            text: The review text to analyze

        Returns:
            Dictionary with 'products' and 'brands' lists
        """
        if not text or not isinstance(text, str):
            return {'products': [], 'brands': []}

        doc = self.nlp(text)

        products = set()
        brands = set()

        for ent in doc.ents:
            ent_text = ent.text.strip()
            ent_lower = ent_text.lower()

            if ent.label_ == 'PRODUCT':
                products.add(ent_text)
            elif ent.label_ == 'ORG':
                brands.add(ent_text)
            elif ent.label_ in ['PERSON', 'GPE']:
                if any(keyword in ent_lower for keyword in self.product_keywords):
                    products.add(ent_text)

        for token in doc:
            token_lower = token.text.lower()
            if token_lower in self.product_keywords and token.pos_ == 'NOUN':
                noun_chunk = self._get_noun_chunk(token)
                if noun_chunk:
                    products.add(noun_chunk)

        for chunk in doc.noun_chunks:
            chunk_text = chunk.text.strip()
            chunk_lower = chunk_text.lower()

            if any(keyword in chunk_lower for keyword in self.product_keywords):
                products.add(chunk_text)

            if chunk.root.pos_ == 'PROPN' and len(chunk_text.split()) <= 3:
                if any(keyword in chunk_lower for keyword in self.product_keywords):
                    products.add(chunk_text)
                else:
                    brands.add(chunk_text)

        return {
            'products': list(products),
            'brands': list(brands)
        }

    def _get_noun_chunk(self, token) -> str:
        """
        Get the noun chunk containing the given token.

        Args:
            token: spaCy token

        Returns:
            Noun chunk text or empty string
        """
        for chunk in token.doc.noun_chunks:
            if token in chunk:
                return chunk.text.strip()
        return ""

    def batch_extract(self, texts: List[str]) -> List[Dict[str, List[str]]]:
        """
        Extract products and brands from multiple texts efficiently.

        Args:
            texts: List of review texts

        Returns:
            List of dictionaries with products and brands for each text
        """
        results = []

        for doc in self.nlp.pipe(texts, batch_size=50):
            text = doc.text
            result = self.extract_products_and_brands(text)
            results.append(result)

        return results



#SENTIMENT ANALYZER
from typing import Dict, List, Tuple


class RuleBasedSentimentAnalyzer:
    """
    Performs sentiment analysis using a rule-based approach with predefined word lists.
    """

    def __init__(self):
        self.positive_words = {
            'excellent', 'amazing', 'wonderful', 'great', 'fantastic', 'awesome',
            'superb', 'outstanding', 'perfect', 'love', 'loved', 'best', 'brilliant',
            'good', 'nice', 'beautiful', 'happy', 'satisfied', 'recommend', 'worth',
            'quality', 'impressive', 'delighted', 'pleased', 'enjoy', 'enjoyed',
            'comfortable', 'easy', 'useful', 'helpful', 'reliable', 'sturdy',
            'durable', 'fast', 'quick', 'efficient', 'powerful', 'clear', 'bright',
            'smooth', 'solid', 'strong', 'better', 'improved', 'upgrade', 'premium',
            'professional', 'convenient', 'attractive', 'elegant', 'sleek', 'modern'
        }

        self.negative_words = {
            'bad', 'terrible', 'horrible', 'awful', 'poor', 'worst', 'hate',
            'hated', 'disappointed', 'disappointing', 'useless', 'waste', 'cheap',
            'broken', 'defective', 'faulty', 'fail', 'failed', 'problem', 'issue',
            'difficult', 'hard', 'complicated', 'confusing', 'frustrating', 'annoying',
            'slow', 'weak', 'flimsy', 'fragile', 'uncomfortable', 'painful',
            'regret', 'returned', 'return', 'refund', 'money back', 'not worth',
            'overpriced', 'expensive', 'scam', 'fraud', 'fake', 'counterfeit',
            'misleading', 'inaccurate', 'unreliable', 'unstable', 'crash', 'crashes',
            'bug', 'bugs', 'error', 'errors', 'malfunction', 'stopped working'
        }

        self.negation_words = {
            'not', 'no', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
            'none', 'hardly', 'scarcely', 'barely', "n't", 'cannot', 'cant',
            'without', 'lack', 'lacks', 'lacking'
        }

        self.intensifiers = {
            'very', 'extremely', 'absolutely', 'really', 'truly', 'incredibly',
            'exceptionally', 'remarkably', 'utterly', 'completely', 'totally',
            'entirely', 'thoroughly', 'highly', 'super', 'quite', 'rather',
            'pretty', 'so', 'too'
        }

    def analyze(self, text: str) -> Dict[str, any]:
        """
        Analyze sentiment of the given text using rule-based approach.

        Args:
            text: The review text to analyze

        Returns:
            Dictionary containing sentiment, confidence_score, positive_words, and negative_words
        """
        if not text or not isinstance(text, str):
            return {
                'sentiment': 'neutral',
                'confidence_score': 0.0,
                'positive_words': [],
                'negative_words': []
            }

        text_lower = text.lower()
        words = text_lower.split()

        positive_found = []
        negative_found = []
        positive_score = 0
        negative_score = 0

        for i, word in enumerate(words):
            word_clean = word.strip('.,!?;:"\'')

            has_negation = False
            if i > 0:
                prev_word = words[i - 1].strip('.,!?;:"\'')
                if prev_word in self.negation_words:
                    has_negation = True

            has_intensifier = False
            if i > 0:
                prev_word = words[i - 1].strip('.,!?;:"\'')
                if prev_word in self.intensifiers:
                    has_intensifier = True

            weight = 1.5 if has_intensifier else 1.0

            if word_clean in self.positive_words:
                if has_negation:
                    negative_score += weight
                    negative_found.append(word_clean)
                else:
                    positive_score += weight
                    positive_found.append(word_clean)

            if word_clean in self.negative_words:
                if has_negation:
                    positive_score += weight
                    positive_found.append(word_clean)
                else:
                    negative_score += weight
                    negative_found.append(word_clean)

        total_score = positive_score + negative_score

        if total_score == 0:
            sentiment = 'neutral'
            confidence = 0.5
        elif positive_score > negative_score:
            sentiment = 'positive'
            confidence = min(positive_score / max(total_score, 1), 1.0)
        elif negative_score > positive_score:
            sentiment = 'negative'
            confidence = min(negative_score / max(total_score, 1), 1.0)
        else:
            sentiment = 'neutral'
            confidence = 0.5

        return {
            'sentiment': sentiment,
            'confidence_score': round(confidence, 3),
            'positive_words': list(set(positive_found)),
            'negative_words': list(set(negative_found))
        }

    def batch_analyze(self, texts: List[str]) -> List[Dict[str, any]]:
        """
        Analyze sentiment for multiple texts.

        Args:
            texts: List of review texts

        Returns:
            List of sentiment analysis results
        """
        return [self.analyze(text) for text in texts]

    def analyze_with_rating(self, text: str, rating: int = None) -> Dict[str, any]:
        """
        Analyze sentiment with optional rating validation.

        Args:
            text: The review text to analyze
            rating: Optional star rating (1-5)

        Returns:
            Dictionary with sentiment analysis and rating consistency
        """
        result = self.analyze(text)

        if rating is not None:
            expected_sentiment = 'positive' if rating >= 4 else ('negative' if rating <= 2 else 'neutral')
            result['rating'] = rating
            result['rating_consistent'] = (result['sentiment'] == expected_sentiment)

        return result



#DATABASE HANDLER
import os
from typing import List, Dict


class DatabaseHandler:
    """
    Handles all database operations for storing reviews and analysis results.
    Falls back to an in-memory store when Supabase credentials or package are missing.
    """

    def __init__(self):
        # try to load env vars (if .env exists)
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass

        supabase_url = os.getenv("VITE_SUPABASE_URL")
        supabase_key = os.getenv("VITE_SUPABASE_ANON_KEY")

        # Try to import supabase client only when credentials are provided
        self._in_memory = True
        self._reviews: List[Dict] = []
        self._entities: List[Dict] = []
        self._sentiments: List[Dict] = []

        if supabase_url and supabase_key:
            try:
                from supabase import create_client, Client  # local import to avoid ImportError at module load
                self.client = create_client(supabase_url, supabase_key)
                self._in_memory = False
                print("Connected to Supabase database.")
            except Exception as e:
                print(f"Warning: Could not connect to Supabase ({e}). Using in-memory storage.")
                self.client = None
                self._in_memory = True
        else:
            print("Warning: Supabase credentials not found. Running in in-memory mode.")
            self.client = None
            self._in_memory = True

    def insert_review(self, review_data: Dict) -> str:
        if self._in_memory:
            import uuid
            record = {
                "id": str(uuid.uuid4()),
                "review_id": review_data.get("review_id"),
                "product_name": review_data.get("product_name", ""),
                "review_text": review_data.get("review_text"),
                "rating": review_data.get("rating"),
            }
            self._reviews.append(record)
            return record["id"]

        data = {
            "review_id": review_data.get("review_id"),
            "product_name": review_data.get("product_name", ""),
            "review_text": review_data.get("review_text"),
            "rating": review_data.get("rating"),
        }

        response = self.client.table("amazon_reviews").insert(data).execute()
        if response.data and len(response.data) > 0:
            return response.data[0]["id"]
        else:
            raise Exception("Failed to insert review into Supabase")

    def insert_entities(self, review_uuid: str, entities: List[Dict]):
        if not entities:
            return

        entity_records = [
            {
                "review_id": review_uuid,
                "entity_type": entity["entity_type"],
                "entity_text": entity["entity_text"],
            }
            for entity in entities
        ]

        if self._in_memory:
            self._entities.extend(entity_records)
            return

        self.client.table("extracted_entities").insert(entity_records).execute()

    def insert_sentiment(self, review_uuid: str, sentiment_data: Dict):
        data = {
            "review_id": review_uuid,
            "sentiment": sentiment_data.get("sentiment"),
            "confidence_score": sentiment_data.get("confidence_score"),
            "positive_words": sentiment_data.get("positive_words", []),
            "negative_words": sentiment_data.get("negative_words", []),
        }

        if self._in_memory:
            record = data.copy()
            record["id"] = str(len(self._sentiments) + 1)
            self._sentiments.append(record)
            return

        self.client.table("sentiment_analysis").insert(data).execute()

    def get_reviews_with_analysis(self, limit: int = 10):
        if self._in_memory:
            results = []
            for review in self._reviews[:limit]:
                sentiments = [s for s in self._sentiments if s.get("review_id") == review["id"]]
                review_copy = review.copy()
                review_copy["sentiment_analysis"] = sentiments
                results.append(review_copy)
            return results

        response = (
            self.client.table("amazon_reviews")
            .select("*, sentiment_analysis(*)")
            .limit(limit)
            .execute()
        )
        return response.data

    def get_entities_by_review(self, review_uuid: str):
        if self._in_memory:
            return [e for e in self._entities if e.get("review_id") == review_uuid]

        response = (
            self.client.table("extracted_entities")
            .select("*")
            .eq("review_id", review_uuid)
            .execute()
        )
        return response.data

    def get_sentiment_statistics(self):
        if self._in_memory:
            stats = {"positive": 0, "negative": 0, "neutral": 0}
            for s in self._sentiments:
                sentiment = s.get("sentiment")
                if sentiment in stats:
                    stats[sentiment] += 1
            return stats

        response = self.client.table("sentiment_analysis").select("sentiment").execute()
        if not response.data:
            return {"positive": 0, "negative": 0, "neutral": 0}

        stats = {"positive": 0, "negative": 0, "neutral": 0}
        for record in response.data:
            sentiment = record.get("sentiment")
            if sentiment in stats:
                stats[sentiment] += 1
        return stats

    def review_exists(self, review_id: str) -> bool:
        if self._in_memory:
            return any(r.get("review_id") == review_id for r in self._reviews)

        response = (
            self.client.table("amazon_reviews")
            .select("id")
            .eq("review_id", review_id)
            .maybeSingle()
            .execute()
        )
        return response.data is not None

if __name__ == "__main__":
    main()



AMAZON REVIEWS SENTIMENT ANALYSIS


Initializing components...
Loaded spaCy model: en_core_web_sm
Components initialized successfully!



Automatically processing 50 reviews...
Kaggle authentication configured successfully.
Dataset URL: https://www.kaggle.com/datasets/bittlingmayer/amazonreviews
Dataset downloaded successfully to data


C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 47%|████▋     | 232M/493M [01:15<01:25, 3.21MB/s] 


KeyboardInterrupt: 